# Lab 6 — ETL Medallion com Parquet

> Extraído de `aulas/Aula_5_etl_medallion_delta.ipynb` (seções 1 a 6 e exercícios 1 a 3; Delta Lake está no Lab 7). As referências a números de seção usam a numeração do notebook original.

# Aula 5 — Pipeline ETL com arquitetura Medallion e Delta Lake
### De arquivos brutos a tabelas confiáveis

Até agora trabalhamos com dados limpos. Na prática, dados chegam **sujos, duplicados, atrasados e em formatos diferentes**. Hoje construímos um pipeline completo que transforma arquivos brutos em tabelas prontas para análise.

**Objetivos**
- Entender data lake, lakehouse e arquitetura Medallion (Bronze, Silver, Gold)
- Ingerir CSV e JSON com problemas reais
- Aplicar regras de qualidade e separar registros rejeitados (quarentena)
- Escrever Parquet particionado e aproveitar *partition pruning*
- Usar Delta Lake: ACID, MERGE, time travel, schema enforcement, OPTIMIZE e VACUUM

---
## 1. Teoria

### Data warehouse, data lake e lakehouse

| | Data Warehouse | Data Lake | Lakehouse |
|-|----------------|-----------|-----------|
| Armazena | Tabelas estruturadas | Arquivos de qualquer tipo | Arquivos abertos + camada de tabela |
| Formato | Proprietário | CSV, JSON, Parquet... | Parquet + log de transações (Delta, Iceberg, Hudi) |
| Custo | Alto | Baixo (object storage) | Baixo |
| Transações (ACID) | Sim | Não | Sim |
| UPDATE / DELETE | Sim | Difícil (reescrever arquivos) | Sim |

O lakehouse junta o custo baixo do data lake com as garantias de um warehouse.

### Arquitetura Medallion

```
  FONTES             BRONZE                SILVER                 GOLD
  CSV, JSON   ──▶   dado bruto     ──▶    dado limpo     ──▶    dado agregado
  APIs, CDC         como chegou           tipado, validado       pronto para
                    + metadados           sem duplicatas         BI / ML / API
                    tudo string           regras de negócio
                                     └──▶ QUARENTENA (rejeitados)
```

| Camada | Pergunta que responde | Regra de ouro |
|--------|----------------------|---------------|
| **Bronze** | "O que exatamente chegou?" | Nunca descartar nada. Permite reprocessar. |
| **Silver** | "Qual é a versão correta de cada registro?" | Tipos certos, uma linha por entidade, regras aplicadas. |
| **Gold** | "O que o negócio precisa ver?" | Agregações e métricas, modeladas para o consumo. |

### Por que Parquet

| | CSV / JSON | Parquet |
|-|-----------|---------|
| Orientação | Linha | **Coluna** |
| Schema | Inferido a cada leitura | Gravado no arquivo |
| Compressão | Fraca | Forte (por coluna) |
| Ler 2 de 20 colunas | Lê tudo | Lê só as 2 (*column pruning*) |
| Filtrar `price > 100` | Lê tudo | Pula blocos via estatísticas min/max (*predicate pushdown*) |

---
## 2. Setup

> Esta aula usa o Delta Lake. No Colab ou localmente:
> `pip install pyspark==4.0.1 delta-spark==4.0.0`
> A primeira criação da sessão baixa os jars do Delta do Maven (precisa de internet).

In [ ]:
# !pip install -q pyspark==4.0.1 delta-spark==4.0.0

In [ ]:
import os, json, csv, random, shutil
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (
    SparkSession.builder
    .appName("Aula 5 - ETL Medallion + Delta")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()   # baixa os jars do Delta na 1a vez
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
LAKE       = "/tmp/lake"
RAW        = f"{LAKE}/raw"
BRONZE     = f"{LAKE}/bronze/orders"
SILVER     = f"{LAKE}/silver/orders"
QUARANTINE = f"{LAKE}/quarantine/orders"
GOLD       = f"{LAKE}/gold"

def tree(path):
    """Mostra a estrutura de pastas com número de arquivos e tamanho."""
    for root, dirs, files in sorted(os.walk(path)):
        dirs.sort()
        data = [f for f in files if not f.startswith((".", "_"))]
        depth = root[len(path):].count(os.sep)
        size = sum(os.path.getsize(os.path.join(root, f)) for f in data) / 1024
        info = f"  ({len(data)} arquivos, {size:,.0f} KB)" if data else ""
        print("    " * depth + (os.path.basename(root) or root) + "/" + info)

def scan_info(df):
    """Mostra os filtros e colunas que chegam até a leitura dos arquivos."""
    import io, contextlib, re
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    scan = next(l for l in buf.getvalue().splitlines() if "FileScan" in l)
    for key in ["PartitionFilters", "PushedFilters", "ReadSchema"]:
        m = re.search(key + r": (\[.*?\]|struct<.*?>)", scan)
        print(f"{key:<17}: {m.group(1) if m else '-'}")

---
## 3. Dados brutos

Simulamos dois sistemas de origem de uma loja:

| Origem | Formato | Particularidades |
|--------|---------|------------------|
| Loja física (PDV) | CSV | Datas em dois formatos, preço com "R$" |
| App de e-commerce | JSON lines | Coluna extra `device`, datas ISO |

Problemas inseridos de propósito: regiões com maiúsculas/espaços inconsistentes ou não informadas, preços inválidos (`N/A`, negativos), quantidades zeradas ou vazias, datas corrompidas, clientes vazios, status fora do domínio, produtos inexistentes, **linhas duplicadas** e **atualizações de status** do mesmo pedido chegando em dias posteriores.

In [ ]:
REGIONS  = ["North", "Northeast", "Midwest", "Southeast", "South"]
STATUSES = ["created", "paid", "shipped", "delivered", "cancelled"]
PRODUCTS = [
    (1, "Notebook", "Computers", 3500), (2, "Monitor", "Computers", 1200),
    (3, "Mouse", "Peripherals", 85),    (4, "Keyboard", "Peripherals", 150),
    (5, "Webcam", "Peripherals", 320),  (6, "USB Hub", "Peripherals", 120),
    (7, "Headset", "Audio", 250),       (8, "Speaker", "Audio", 400),
    (9, "External HD", "Storage", 400), (10, "Pendrive", "Storage", 60),
    (11, "USB Cable", "Cables", 25),    (12, "HDMI Cable", "Cables", 40),
]

rng = random.Random(42)
next_id = [100_000]
known_orders = {}   # order_id -> created_at (para gerar atualizações em dias seguintes)

def dirty_region(r):
    if rng.random() < 0.01: return "N/D"           # região não informada
    return rng.choice([r, r, r, r.lower(), " " + r.upper(), r + "  "])

def dirty_price(p):
    roll = rng.random()
    if roll < 0.03: return "N/A"
    if roll < 0.05: return f"-{p:.2f}"
    if roll < 0.25: return f"R$ {p:.2f}"
    return f"{p:.2f}"

def dirty_quantity():
    roll = rng.random()
    if roll < 0.02: return ""
    if roll < 0.04: return "0"
    return str(rng.randint(1, 4))

def make_order(order_id, created, updated, status, fmt):
    pid, _, _, base = rng.choice(PRODUCTS)
    if rng.random() < 0.02:
        pid = 999                                   # produto inexistente
    ts_created = created.strftime(fmt) if rng.random() > 0.01 else "2024-13-45 99:99"   # data corrompida
    return {
        "order_id":    str(order_id),
        "customer_id": str(rng.randint(1, 3000)) if rng.random() > 0.01 else "",
        "product_id":  str(pid),
        "quantity":    dirty_quantity(),
        "price":       dirty_price(base * rng.uniform(0.9, 1.1)),
        "region":      dirty_region(rng.choice(REGIONS)),
        "status":      status if rng.random() > 0.01 else "pending_review",       # status fora do domínio
        "created_at":  ts_created,
        "updated_at":  updated.strftime(fmt),
    }

def generate_day(day, n_new=1200, n_updates=300):
    """Gera os arquivos de um dia: pedidos novos + atualizações de pedidos antigos."""
    rows_csv, rows_json = [], []
    for _ in range(n_new):
        oid = next_id[0]; next_id[0] += 1
        created = day + timedelta(minutes=rng.randint(0, 1439))
        is_app = rng.random() < 0.6
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else rng.choice(["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"])
        row = make_order(oid, created, created, rng.choice(["created", "paid"]), fmt)
        known_orders[oid] = (created, is_app, row)
        (rows_json if is_app else rows_csv).append(row)

    # atualizações de status de pedidos de dias anteriores
    old = [o for o, (c, _, _) in known_orders.items() if c.date() < day.date()]
    for oid in rng.sample(old, min(n_updates, len(old))):
        created, is_app, original = known_orders[oid]
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else "%Y-%m-%d %H:%M:%S"
        row = dict(original)
        row["status"] = rng.choice(["shipped", "delivered", "cancelled"])
        row["created_at"] = created.strftime(fmt)
        row["updated_at"] = (day + timedelta(minutes=rng.randint(0, 1439))).strftime(fmt)
        (rows_json if is_app else rows_csv).append(row)

    # duplicatas exatas (reenvio de arquivo, retry de API...)
    rows_csv  += rng.sample(rows_csv,  len(rows_csv)  // 30)
    rows_json += rng.sample(rows_json, len(rows_json) // 30)

    tag = day.strftime("%Y%m%d")
    with open(f"{RAW}/orders_csv/pdv_{tag}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows_csv[0].keys()))
        w.writeheader(); w.writerows(rows_csv)
    with open(f"{RAW}/orders_json/app_{tag}.json", "w") as f:
        for r in rows_json:
            r = dict(r, device=rng.choice(["ios", "android", "web"]))
            if rng.random() < 0.5 and r["price"].replace(".", "").isdigit():
                r["price"] = float(r["price"])      # no JSON, às vezes número, às vezes texto
            f.write(json.dumps(r) + "\n")
    return len(rows_csv), len(rows_json)

shutil.rmtree(LAKE, ignore_errors=True)
os.makedirs(f"{RAW}/orders_csv"); os.makedirs(f"{RAW}/orders_json")

with open(f"{RAW}/products.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["product_id", "product_name", "category", "list_price"]); w.writerows(PRODUCTS)

for d in ["2024-07-01", "2024-07-02", "2024-07-03"]:
    n_csv, n_json = generate_day(datetime.fromisoformat(d))
    print(f"{d}: {n_csv} linhas CSV, {n_json} linhas JSON")

tree(RAW)

In [ ]:
# Uma olhada nos arquivos como eles são
print(open(f"{RAW}/orders_csv/pdv_20240701.csv").read()[:600])
print(open(f"{RAW}/orders_json/app_20240701.json").readline())

---
## 4. Bronze: ingerir sem perder nada

Regras da camada Bronze:
- Ler **tudo como string**. Se um preço vier como `N/A`, a leitura não pode falhar nem transformar o valor em nulo silenciosamente.
- Juntar as fontes preservando colunas que só existem em uma delas (`unionByName(..., allowMissingColumns=True)`).
- Adicionar **metadados de linhagem**: de qual arquivo veio e quando foi ingerido.

> `inferSchema=True` é prático, mas perigoso em pipelines: o schema pode mudar de um dia para o outro conforme os dados que chegam.

In [ ]:
raw_csv = (
    spark.read
    .option("header", True)                 # sem inferSchema: tudo string
    .csv(f"{RAW}/orders_csv")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("store"))
)

raw_json = (
    spark.read
    .option("primitivesAsString", True)     # números e booleanos também viram string
    .json(f"{RAW}/orders_json")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("app"))
)

bronze = (
    raw_csv.unionByName(raw_json, allowMissingColumns=True)
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze.write.mode("overwrite").parquet(BRONZE)
bronze = spark.read.parquet(BRONZE)

bronze.printSchema()
bronze.groupBy("channel").count().show()

`_metadata` é uma coluna oculta que o Spark disponibiliza em leituras de arquivos (caminho, tamanho, data de modificação). Ela substitui a antiga `input_file_name()`.

---
## 5. Silver: limpar, validar e deduplicar

O trabalho da Silver se divide em quatro etapas, cada uma em uma função. Funções pequenas e reutilizáveis permitem processar novos dias com o mesmo código (seção 8).

```
  bronze ──▶ clean() ──▶ enrich() ──▶ validate() ──┬──▶ deduplicate() ──▶ SILVER
             tipos        join com                  │
             formatos     produtos                  └──▶ QUARENTENA (+ motivo)
```

### ANSI mode no Spark 4

No Spark 4, o **modo ANSI vem ligado por padrão**: `cast("abc" as int)` **gera erro** em vez de retornar nulo. Em dados sujos, use as variantes `try_`:

| Operação | Falha com valor inválido | Retorna nulo |
|----------|-------------------------|--------------|
| Conversão de tipo | `col.cast("int")` | `col.try_cast("int")` |
| Texto para timestamp | `F.to_timestamp(c, fmt)` | `F.try_to_timestamp(c, F.lit(fmt))` |
| Divisão | `a / b` | `F.try_divide(a, b)` |

In [ ]:
# Demonstração do ANSI mode
demo = spark.createDataFrame([("10",), ("N/A",)], ["v"])
demo.select("v", F.col("v").try_cast("int").alias("try_cast")).show()

import logging
quiet = logging.getLogger("DataFrameQueryContextLogger")
spark.sparkContext.setLogLevel("OFF"); quiet.setLevel(logging.CRITICAL)   # só para não poluir o output

try:
    demo.select(F.col("v").cast("int")).show()
except Exception as e:
    print("cast() falhou:", str(e).split(".")[0])

spark.sparkContext.setLogLevel("ERROR"); quiet.setLevel(logging.NOTSET)

In [ ]:
TS_FORMATS = ["yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm", "yyyy-MM-dd'T'HH:mm:ss"]

def parse_ts(col):
    """Tenta cada formato e fica com o primeiro que funcionar."""
    return F.coalesce(*[F.try_to_timestamp(F.col(col), F.lit(fmt)) for fmt in TS_FORMATS])

def clean(df):
    return (
        df
        .withColumn("order_id",    F.col("order_id").try_cast("long"))
        .withColumn("customer_id", F.col("customer_id").try_cast("long"))
        .withColumn("product_id",  F.col("product_id").try_cast("int"))
        .withColumn("quantity",    F.col("quantity").try_cast("int"))
        .withColumn("price",       F.regexp_replace("price", r"R\$|\s", "").try_cast("double"))
        .withColumn("region",      F.initcap(F.trim("region")))
        .withColumn("status",      F.lower(F.trim("status")))
        .withColumn("created_at",  parse_ts("created_at"))
        .withColumn("updated_at",  parse_ts("updated_at"))
    )

products = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW}/products.csv")

def enrich(df):
    return (
        df.join(F.broadcast(products.select("product_id", "product_name", "category")), "product_id", "left")
        .withColumn("revenue", F.round(F.col("price") * F.col("quantity"), 2))
        .withColumn("order_date", F.to_date("created_at"))
    )

clean(bronze).select("order_id", "price", "quantity", "region", "created_at").show(5)

### Regras de qualidade e quarentena

Em vez de apagar registros ruins com `filter`, guardamos os rejeitados **com o motivo**. Assim é possível medir a qualidade da fonte, avisar o time de origem e reprocessar depois de uma correção.

In [ ]:
RULES = {
    "order_id_nulo":        F.col("order_id").isNull(),
    "preco_invalido":       F.col("price").isNull() | (F.col("price") <= 0),
    "quantidade_invalida":  F.col("quantity").isNull() | (F.col("quantity") <= 0),
    "data_invalida":        F.col("created_at").isNull() | F.col("updated_at").isNull(),
    "regiao_desconhecida":  F.col("region").isNull() | ~F.col("region").isin(REGIONS),
    "produto_desconhecido": F.col("category").isNull(),
}

def validate(df):
    errors = F.array_compact(F.array(*[F.when(cond, F.lit(name)) for name, cond in RULES.items()]))
    df = df.withColumn("_errors", errors)
    valid    = df.filter(F.size("_errors") == 0).drop("_errors")
    rejected = df.filter(F.size("_errors") > 0)
    return valid, rejected

def deduplicate(df):
    """Mantém só a versão mais recente de cada pedido."""
    w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(), F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

SILVER_COLS = ["order_id", "customer_id", "product_id", "product_name", "category",
               "quantity", "price", "revenue", "region", "status", "channel", "device",
               "created_at", "updated_at", "order_date", "_source_file", "_ingested_at"]

valid, rejected = validate(enrich(clean(bronze)))
silver_df = deduplicate(valid).select(SILVER_COLS)

print(f"Bronze    : {bronze.count():>6,} linhas")
print(f"Válidas   : {valid.count():>6,} linhas")
print(f"Rejeitadas: {rejected.count():>6,} linhas")
print(f"Silver    : {silver_df.count():>6,} pedidos únicos")

In [ ]:
# Por que os registros foram rejeitados?
(
    rejected
    .select(F.explode("_errors").alias("motivo"))
    .groupBy("motivo").count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

### Escrita particionada

`partitionBy("order_date")` cria uma pasta por valor de data. Consultas que filtram por data leem apenas as pastas necessárias (*partition pruning*).

Boas colunas de partição têm **baixa cardinalidade** e são **usadas em filtros** (data, região). Particionar por `order_id` criaria milhões de pastas minúsculas.

In [ ]:
(silver_df
 .repartition(8)                     # simula um job com 8 tasks no último stage
 .write.mode("overwrite")
 .partitionBy("order_date")
 .parquet(SILVER))

rejected.write.mode("overwrite").parquet(QUARANTINE)

tree(f"{LAKE}/silver")

Repare no número de arquivos por partição. Cada task do último stage escreve **um arquivo por partição de data que encontrar**. Com 8 tasks, surgem até 8 arquivos por dia. Esse é o problema dos **arquivos pequenos**: milhares de arquivos de poucos KB deixam a leitura lenta.

A correção é organizar os dados por partição antes de escrever:

In [ ]:
(silver_df
 .repartition("order_date")          # todas as linhas de um dia na mesma task
 .write.mode("overwrite")
 .partitionBy("order_date")
 .parquet(SILVER))

tree(f"{LAKE}/silver")

### Partition pruning, predicate pushdown e column pruning

Os três mecanismos aparecem no plano de leitura (`FileScan parquet`):

| Mecanismo | Campo no plano | Efeito |
|-----------|----------------|--------|
| Partition pruning | `PartitionFilters` | Pula pastas inteiras |
| Predicate pushdown | `PushedFilters` | Pula blocos dentro dos arquivos |
| Column pruning | `ReadSchema` | Lê só as colunas usadas |

In [ ]:
silver = spark.read.parquet(SILVER)

q = (silver
     .filter(F.col("order_date") == "2024-07-02")   # coluna de partição
     .filter(F.col("price") > 1000)                 # coluna comum
     .select("order_id", "price", "region"))

scan_info(q)

### Reprocessar só uma partição

Se o dia 03/07 precisar ser reprocessado, não queremos reescrever a tabela inteira. Com `partitionOverwriteMode=dynamic`, o `overwrite` substitui **apenas as partições presentes no DataFrame**.

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

before = spark.read.parquet(SILVER).groupBy("order_date").count().orderBy("order_date")
before.show()

day3 = silver_df.filter(F.col("order_date") == "2024-07-03").limit(100)   # simula um reprocessamento
day3.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(SILVER)

spark.read.parquet(SILVER).groupBy("order_date").count().orderBy("order_date").show()

# volta ao estado completo
silver_df.repartition("order_date").write.mode("overwrite").partitionBy("order_date").parquet(SILVER)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

> Com o modo padrão (`static`), o mesmo comando teria **apagado os dias 01 e 02**. Esse é um dos acidentes mais comuns em pipelines com Parquet puro.

---
## 6. Gold: tabelas para o negócio

A Gold responde perguntas específicas. Pedidos cancelados não entram na receita.

In [ ]:
silver = spark.read.parquet(SILVER)
sales  = silver.filter(F.col("status") != "cancelled")

gold_daily = (
    sales.groupBy("order_date", "region", "category")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("revenue"), 2).alias("revenue"),
        F.round(F.avg("revenue"), 2).alias("avg_ticket"),
    )
)

gold_products = (
    sales.groupBy("product_name", "category")
    .agg(F.sum("quantity").alias("units"), F.round(F.sum("revenue"), 2).alias("revenue"))
    .withColumn("rank", F.dense_rank().over(Window.orderBy(F.desc("revenue"))))
)

gold_daily.write.mode("overwrite").partitionBy("order_date").parquet(f"{GOLD}/daily_sales")
gold_products.write.mode("overwrite").parquet(f"{GOLD}/product_ranking")

spark.read.parquet(f"{GOLD}/product_ranking").orderBy("rank").show(5)
tree(LAKE)

---
## 9. Exercícios

In [ ]:
# ── Exercício 1 ─────────────────────────────────────────────────────────────
# Adicione duas regras de qualidade ao dicionário RULES:
#   - "cliente_nulo": customer_id nulo
#   - "status_invalido": status fora de STATUSES
# Rode o validate novamente e mostre quantas linhas cada regra rejeita.
# Dica: uma linha pode violar mais de uma regra.

# Sua solução:

In [ ]:
# ── Exercício 2 ─────────────────────────────────────────────────────────────
# Crie uma tabela Gold "funil de status por canal":
#   para cada channel e status, o número de pedidos e o percentual dentro do canal.
# Salve em f"{GOLD}/status_funnel".

# Sua solução:

In [ ]:
# ── Exercício 3 ─────────────────────────────────────────────────────────────
# Escreva gold_daily particionado por "region" e prove, pelo plano de execução,
# que uma consulta com filtro region = 'South' faz partition pruning.

# Sua solução:

### Soluções

In [ ]:
# ── Solução 1 ───────────────────────────────────────────────────────────────
RULES_V2 = {
    **RULES,
    "cliente_nulo":    F.col("customer_id").isNull(),
    "status_invalido": F.col("status").isNull() | ~F.col("status").isin(STATUSES),
}

def validate_v2(df):
    errors = F.array_compact(F.array(*[F.when(c, F.lit(n)) for n, c in RULES_V2.items()]))
    df = df.withColumn("_errors", errors)
    return df.filter(F.size("_errors") == 0).drop("_errors"), df.filter(F.size("_errors") > 0)

_, rejected_v2 = validate_v2(enrich(clean(bronze)))
(rejected_v2.select(F.explode("_errors").alias("motivo"))
 .groupBy("motivo").count().orderBy(F.desc("count")).show(truncate=False))

In [ ]:
# ── Solução 2 ───────────────────────────────────────────────────────────────
funnel = (
    spark.read.parquet(SILVER)
    .groupBy("channel", "status").count()
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(Window.partitionBy("channel")) * 100, 1))
    .orderBy("channel", F.desc("count"))
)
funnel.write.mode("overwrite").parquet(f"{GOLD}/status_funnel")
funnel.show()

In [ ]:
# ── Solução 3 ───────────────────────────────────────────────────────────────
gold_daily.write.mode("overwrite").partitionBy("region").parquet(f"{GOLD}/daily_by_region")

q = spark.read.parquet(f"{GOLD}/daily_by_region").filter(F.col("region") == "South")
scan_info(q)
tree(f"{GOLD}/daily_by_region")

In [ ]:
spark.stop()